# AutoPilot Studio — Premium Watch Reel Renderer

**Goal:** generate a short vertical montage from real AI-generated moving shots using the open Wan2.1 1.3B text-to-video model. This is an experimental render pipeline, not a promise of client-ready quality. Model execution has not been verified in this repository because a free GPU is not guaranteed.

## Safety and cost gates
- Run only on a free GPU runtime if one is offered. If no GPU is available, stop; do not upgrade or enter payment details.
- The model download is several GB and each shot can take a long time.
- Review each shot for melting objects, flicker, inconsistent watch design, and false product claims before using it commercially.
- This uses a fictional, unbranded watch concept. Do not imply actual water resistance, warranty, materials, or product specifications without evidence.
- The final video is a first-pass motion prototype. Premium/client-ready status must be decided after visual review.

In [ ]:
import os, sys, subprocess, json, textwrap, shutil
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError('No NVIDIA GPU detected. Stop here; free GPU access is not guaranteed. Do not pay to continue.')
print(gpu.stdout[:2000])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers>=0.31.0', 'transformers>=4.49.0', 'tokenizers>=0.20.3', 'accelerate>=1.1.1', 'opencv-python>=4.9.0.80', 'tqdm', 'imageio', 'easydict', 'ftfy', 'imageio-ffmpeg', 'huggingface_hub', 'numpy<2'], check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('PyTorch cannot access CUDA. Stop; do not pay for compute.')
print('GPU:', torch.cuda.get_device_name(0), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
repo='/content/Wan2.1'
if not os.path.isdir(repo): subprocess.run(['git','clone','--depth','1','https://github.com/Wan-Video/Wan2.1.git',repo],check=True)
from huggingface_hub import snapshot_download
model='/content/Wan2.1-T2V-1.3B'
snapshot_download(repo_id='Wan-AI/Wan2.1-T2V-1.3B',local_dir=model)
outdir='/content/autopilot_watch_shots'; os.makedirs(outdir,exist_ok=True)
print('Model and inference code ready.')

In [ ]:
shots = [
  ('01_macro', 'One uninterrupted cinematic macro shot of a fictional unbranded luxury mechanical wristwatch movement in a black studio. Tiny gold gears visibly rotate at different speeds, a balance wheel oscillates continuously, sapphire reflections travel across brushed steel, slow controlled camera orbit, shallow depth of field, photorealistic premium product cinematography, no text, no logo.'),
  ('02_exploded', 'One uninterrupted photorealistic CGI shot of the components of a fictional luxury mechanical watch floating in precise alignment in a dark studio. The bezel, sapphire crystal, dial, tiny gears and steel case move slowly toward each other and assemble with believable mechanical motion, warm gold edge lights, cool black-blue shadows, crisp reflections, slow camera push, no text, no logo.'),
  ('03_wrist', 'One uninterrupted premium live-action-style close-up of a person fastening a fictional black and gold mechanical watch onto their wrist in a dark luxury studio. Natural hand movement, realistic skin and metal reflections, subtle camera orbit, controlled highlights, shallow depth of field, cinematic commercial, no text, no logo.'),
  ('04_hero', 'One uninterrupted cinematic product hero shot of a fictional black and gold mechanical wristwatch standing on wet dark volcanic rock at sunrise, mist drifting past, tiny droplets moving on the steel, warm light sweeping over the dial, dramatic distant mountains, slow dolly-in, realistic luxury product photography, no text, no logo.')
]
negative='still image, slideshow, static object, blurry, low resolution, flicker, jitter, melting metal, malformed watch, duplicated object, extra hands, fake text, logo, watermark, sudden camera shake, oversaturated highlights'
for name,prompt in shots:
    path=f'{outdir}/{name}.mp4'
    if os.path.exists(path) and os.path.getsize(path)>100_000:
        print('Keeping existing shot:',path); continue
    cmd=[sys.executable,'generate.py','--task','t2v-1.3B','--size','480*832','--ckpt_dir',model,'--offload_model','True','--t5_cpu','--sample_shift','8','--sample_guide_scale','6','--frame_num','49','--save_file',path,'--prompt',prompt+' Negative prompt: '+negative]
    print('Generating actual moving shot:',name)
    subprocess.run(cmd,cwd=repo,check=True)
    if not os.path.isfile(path) or os.path.getsize(path)<100_000: raise RuntimeError('Shot failed quality/file-size sanity check: '+name)
print('All four shot files exist. This does not mean they passed visual quality review.')

In [ ]:
# Normalize video properties, then concatenate actual generated clips. No still-image slideshow is used here.
normalized=[]
for name,_ in shots:
    src=f'{outdir}/{name}.mp4'; dst=f'{outdir}/{name}_norm.mp4'
    subprocess.run(['ffmpeg','-y','-i',src,'-vf','scale=480:832:force_original_aspect_ratio=increase,crop=480:832,fps=24,format=yuv420p','-an','-c:v','libx264','-preset','veryfast','-crf','19',dst],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    normalized.append(dst)
concat_file=f'{outdir}/concat.txt'
with open(concat_file,'w') as f:
    for p in normalized: f.write("file '"+p+"'\n")
final='/content/AutoPilot_Watch_Premium_Motion_Prototype.mp4'
subprocess.run(['ffmpeg','-y','-f','concat','-safe','0','-i',concat_file,'-c:v','libx264','-preset','slow','-crf','18','-pix_fmt','yuv420p','-movflags','+faststart',final],check=True)
probe=subprocess.run(['ffprobe','-v','error','-show_entries','format=duration:stream=codec_name,width,height,r_frame_rate','-of','json',final],capture_output=True,text=True,check=True)
print(json.dumps(json.loads(probe.stdout),indent=2))
from IPython.display import Video,display
display(Video(final,embed=True))
from google.colab import files
files.download(final)

## Mandatory visual quality gate
Do not call the reel successful just because an MP4 was produced. Watch the full video at normal speed and inspect individual frames. Reject and retry any shot with obvious morphing, gear movement that makes no physical sense, inconsistent watch identity, broken hands, severe flicker, or muddy detail. This first pass is a **motion prototype**; no audio mix, licensed music, brand-specific product accuracy, or final color grade is included. Only label it premium after it passes a human review.